In [6]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [2]:
# 전처리 완료된 파일 불러오기
E = pd.read_csv('C:/Users/hyeon/OneDrive/Desktop/2026-1/공모전/데이터분석/위험지수산출/E_표면유출위험지수_결과.csv')
S1 = pd.read_csv('C:/Users/hyeon/OneDrive/Desktop/2026-1/공모전/데이터분석/위험지수산출/S1_반지하비율_결과.csv')
S2_3 = pd.read_csv('C:/Users/hyeon/OneDrive/Desktop/2026-1/공모전/데이터분석/위험지수산출/S2_S3_하수도_결과.csv')
AC = pd.read_csv('C:/Users/hyeon/OneDrive/Desktop/2026-1/공모전/데이터분석/위험지수산출/AC_적응능력_결과.csv')

In [4]:
# E 데이터프레임을 기준으로 left join
final_df = pd.merge(E[['자치구', '표면유출_위험지수']], S1[['자치구', 'S1_정규화']], on='자치구', how='left')
final_df = pd.merge(final_df, S2_3[['자치구', 'S2_Vulnerability', 'S3_Vulnerability']], on='자치구', how='left')
final_df = pd.merge(final_df, AC[['자치구', 'AC_최종지수']], on='자치구', how='left')

In [7]:
# PCA 기반 객관적 가중치 산출 및 S지표 통합

# 1. 가중치를 구할 대상 민감도 변수들만 추출
s_features = ['S1_정규화', 'S2_Vulnerability', 'S3_Vulnerability']
s_data = final_df[s_features]

In [8]:
# 2. 스케일링 
scaler = StandardScaler()
s_scaled = scaler.fit_transform(s_data)

In [9]:
# 3. PCA 모델 학습 (데이터의 분산을 가장 잘 설명하는 제1주성분만 추출)
pca = PCA(n_components=1)
pca.fit(s_scaled)

PCA(n_components=1)

In [10]:
# 4. 제1주성분의 적재값 추출 후 가중치로 변환
# 각 지표가 전체 취약성 데이터를 설명하는 데 기여하는 정도의 절댓값을 비율로 환산
loadings = np.abs(pca.components_[0])
weights = loadings / np.sum(loadings)

In [11]:
for col, w in zip(s_features, weights):
    print(f"- {col} 가중치: {w:.4f} ({w*100:.1f}%)")

- S1_정규화 가중치: 0.1777 (17.8%)
- S2_Vulnerability 가중치: 0.4162 (41.6%)
- S3_Vulnerability 가중치: 0.4061 (40.6%)


In [12]:
# 5. 산출된 객관적 가중치를 반영하여 S_통합지수 계산
final_df['S_통합지수'] = (
    final_df['S1_정규화'] * weights[0] + 
    final_df['S2_Vulnerability'] * weights[1] + 
    final_df['S3_Vulnerability'] * weights[2]
)

In [13]:
### 홍수 취약성 공식 (V = E * S - AC)
final_df['V_원시값'] = (final_df['표면유출_위험지수'] * final_df['S_통합지수']) - final_df['AC_최종지수']

In [14]:
# 0 ~ 1 정규화 (가장 취약한 구가 1)
v_min = final_df['V_원시값'].min()
v_max = final_df['V_원시값'].max()
final_df['V_최종지수'] = (final_df['V_원시값'] - v_min) / (v_max - v_min)

In [15]:
# 결과 확인
final_ranking = final_df.sort_values(by='V_최종지수', ascending=False).reset_index(drop=True)
display(final_ranking[['자치구', '표면유출_위험지수', 'S_통합지수', 'AC_최종지수', 'V_최종지수']])

,자치구,표면유출_위험지수,S_통합지수,AC_최종지수,V_최종지수
0,중구,0.864447,0.742938,0.025699,1.000000
1,구로구,0.915145,0.804738,0.264442,0.869171
2,서대문구,0.574714,0.769749,0.014341,0.829370
3,금천구,0.620374,0.731186,0.122348,0.741756
4,양천구,0.796517,0.584113,0.146546,0.730394
5,동작구,0.617971,0.570243,0.072637,0.695130
6,중랑구,0.676437,0.562280,0.121242,0.676435
7,영등포구,0.613890,0.732163,0.241576,0.630074
8,도봉구,0.324141,0.718506,0.026596,0.628634
9,동대문구,0.960891,0.507415,0.359311,0.557986


In [16]:
# '순위' 열 새로 만들기 
final_ranking['순위'] = final_ranking['V_최종지수'].rank(ascending=False, method='min').astype(int)
display_cols = ['순위', '자치구', '표면유출_위험지수', 'S_통합지수', 'AC_최종지수', 'V_최종지수']
display(final_ranking[display_cols])

# 최종 결과물을 CSV로 저장 (순위 열 포함)
final_ranking.to_csv('C:/Users/hyeon/OneDrive/Desktop/2026-1/공모전/데이터분석/위험지수산출/최종_홍수취약성_랭킹.csv', index=False, encoding='utf-8-sig')

,순위,자치구,표면유출_위험지수,S_통합지수,AC_최종지수,V_최종지수
0,1,중구,0.864447,0.742938,0.025699,1.000000
1,2,구로구,0.915145,0.804738,0.264442,0.869171
2,3,서대문구,0.574714,0.769749,0.014341,0.829370
3,4,금천구,0.620374,0.731186,0.122348,0.741756
4,5,양천구,0.796517,0.584113,0.146546,0.730394
5,6,동작구,0.617971,0.570243,0.072637,0.695130
6,7,중랑구,0.676437,0.562280,0.121242,0.676435
7,8,영등포구,0.613890,0.732163,0.241576,0.630074
8,9,도봉구,0.324141,0.718506,0.026596,0.628634
9,10,동대문구,0.960891,0.507415,0.359311,0.557986
